# 05 · Final Load Prep (Tableau-Ready)
**Goal:** Aggregate KPIs and export a dataset ready for Tableau dashboards.

In [1]:
import pandas as pd, numpy as np, os
df = pd.read_csv('../data/processed/cleaned.csv')
print('Loaded:', df.shape)

Loaded: (13000, 36)


### 1 · Compute margin & aggregate

In [2]:
df['cost_price'] = df['final_price'] * (1 - df['profit_margin_pct']/100)
df['margin_inr'] = (df['final_price'] - df['cost_price']) * df['sold_quantity']
agg = df.groupby(['city','category','basket_type','offer_type']).agg(
    revenue=('revenue_inr','sum'),
    margin=('margin_inr','sum'),
    avg_price=('final_price','mean'),
    total_qty=('sold_quantity','sum'),
    impulse_premium=('impulse_premium_pct','mean')
).reset_index()
print('Aggregated shape:', agg.shape)

Aggregated shape: (647, 9)


### 2 · KPIs

In [3]:
agg['margin_pct'] = (agg['margin'] / agg['revenue'] * 100).round(2)
base = agg[agg['offer_type']=='no offer'].groupby(
    ['city','category','basket_type'])['total_qty'].mean().reset_index(name='base_qty')
agg = agg.merge(base, on=['city','category','basket_type'], how='left')
agg['offer_lift'] = np.where(agg['base_qty']>0, agg['total_qty']/agg['base_qty'], 1.0)
tot = agg.groupby(['city','category'])['revenue'].transform('sum')
agg['impulse_pct'] = np.where(
    agg['basket_type']=='Impulse',
    (agg['revenue']/tot*100).round(2), 0)

### 3 · Export

In [4]:
final = agg[['city','category','basket_type','revenue','impulse_pct',
             'avg_price','offer_lift','margin_pct']].copy()
os.makedirs('../data/processed', exist_ok=True)
final.to_csv('../data/processed/final_dataset.csv', index=False)
print('Exported final_dataset.csv ->', final.shape)
print(final.head(10))

Exported final_dataset.csv -> (647, 8)
        city   category basket_type     revenue  impulse_pct   avg_price  \
0  ahmedabad     bakery     Impulse   573090.63         7.78  221.454091   
1  ahmedabad     bakery     Impulse  1089202.42        14.79  260.124444   
2  ahmedabad     bakery     Impulse   981679.10        13.33  231.278125   
3  ahmedabad     bakery     Impulse   357140.64         4.85  199.571111   
4  ahmedabad     bakery     Impulse  2386191.21        32.40  242.201346   
5  ahmedabad     bakery       Mixed  1978075.02         0.00  221.940862   
6  ahmedabad  beverages     Impulse   203763.34         4.82  165.226000   
7  ahmedabad  beverages     Impulse   216734.95         5.13  155.714286   
8  ahmedabad  beverages     Impulse   289606.34         6.85  155.674444   
9  ahmedabad  beverages     Impulse    80809.42         1.91  135.350000   

   offer_lift  margin_pct  
0    0.290536       18.81  
1    0.390281       23.85  
2    0.421575       23.77  
3    0.15004